# Limpiar las víctimas
Objetivo: una tabla con una fila por persona muerta o herida, con los datos de su siniestro.

cargar y juntar las personas

In [1]:
import pandas as pd

carpeta = "../data/raw/Siniestros_viales/"

actores = pd.read_csv(carpeta + "SIGAT_ANUARIO_2024_Actor_vial.csv.gz", low_memory=False)
act_2025 = pd.read_excel(carpeta + "SIGAT_ANUARIO_2025.xlsx", sheet_name="Actor vial")

act_2025 = act_2025.rename(columns={"Codigo_Siniestro": "Codigo_Accidente"})

columnas = ["Codigo_Accidentado", "Codigo_Accidente", "Condicion", "Gravedad_Indicador_30d"]
personas = pd.concat([actores[columnas], act_2025[columnas]], ignore_index=True)
print(personas.shape)

(636184, 4)


Revisar personas repetidas

In [2]:
print(personas["Codigo_Accidentado"].duplicated().sum())
personas = personas.drop_duplicates(subset="Codigo_Accidentado", keep="last")
print(personas.shape)

5
(636179, 4)


Quedarnos con muertos y heridos

In [3]:
print(personas["Gravedad_Indicador_30d"].value_counts())

victimas = personas[personas["Gravedad_Indicador_30d"].isin(["HERIDO", "MUERTO"])]
print(victimas.shape)

Gravedad_Indicador_30d
ILESO     426541
HERIDO    199543
MUERTO      5639
Name: count, dtype: int64
(205182, 4)


Unión con los siniestros

In [4]:
siniestros = pd.read_csv("../data/processed/siniestros.csv.gz")

union = victimas.merge(siniestros, on="Codigo_Accidente", how="left",
                       validate="m:1", indicator=True)
print(union.shape)
print(union["_merge"].value_counts())

(205182, 15)
_merge
both          198030
left_only       7152
right_only         0
Name: count, dtype: int64


Quedarnos con las que tienen pareja y guardar

In [5]:
victimas = union[union["_merge"] == "both"]
victimas = victimas.drop(columns=["_merge", "Gravedad_indicador_30d"])
print(victimas.shape)
print(victimas["Gravedad_Indicador_30d"].value_counts())
print(victimas["Condicion"].value_counts())

victimas.to_csv("../data/processed/victimas.csv.gz", index=False)

(198030, 13)
Gravedad_Indicador_30d
HERIDO    192400
MUERTO      5630
Name: count, dtype: int64
Condicion
MOTOCICLISTA    86008
PEATÓN          37429
PASAJERO        32640
CICLISTA        21308
CONDUCTOR       20645
Name: count, dtype: int64


## Decisiones
- Juntamos las personas de 2015-2024 y 2025: 636.184.
- Quitamos 5 personas repetidas (eran de los 3 siniestros repetidos): quedan 636.179.
- Nos quedamos con los muertos y heridos: 205.182 (199.543 heridos y 5.639 muertos).
- Unimos cada víctima con su siniestro (muchos a uno, validate="m:1").
- 7.152 víctimas quedaron sin pareja: 6.638 heridos cuyo siniestro no tenía coordenadas (2015-2020) y 514 víctimas (505 heridos y 9 muertos) de los siniestros con la coordenada de relleno de 2022.
- Quedan 198.030 víctimas (192.400 heridos y 5.630 muertos).
- No guardamos edad ni sexo porque no los usamos (privacidad).